# YOLOv1 — Training on Google Colab
**Paper:** *You Only Look Once: Unified, Real-Time Object Detection* (CVPR 2016)

**Dataset:** Football Players Detection (ball, goalkeeper, player, referee)

**Steps:**
1. Upload `YOLOv1_Project.zip` to your Google Drive root folder
2. Upload `Football Players Detection Yolov5.zip` to your Google Drive root folder
3. Run all cells in order

> **Important:** Go to `Runtime → Change runtime type → T4 GPU` before running.

## Step 1 — Mount Drive & Extract Files

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

import os, sys, zipfile, shutil

# ── Extract YOLOv1 project code ──────────────────────────────────────────────
PROJECT_ZIP  = '/content/drive/MyDrive/YOLOv1_Project.zip'
PROJECT_DIR  = '/content/yolov1'

if not os.path.exists(PROJECT_DIR):
    print('Extracting project code...')
    with zipfile.ZipFile(PROJECT_ZIP, 'r') as z:
        z.extractall('/content/')
    # Handle nested folder if needed
    extracted = [d for d in os.listdir('/content/') if os.path.isdir(f'/content/{d}') and 'yolo' in d.lower()]
    if extracted and extracted[0] != 'yolov1':
        os.rename(f'/content/{extracted[0]}', PROJECT_DIR)
    print(f'Project extracted to: {PROJECT_DIR}')
else:
    print(f'Project already at: {PROJECT_DIR}')

# ── Extract dataset ──────────────────────────────────────────────────────────
DATASET_ZIP  = '/content/drive/MyDrive/Football Players Detection Yolov5.zip'
DATASET_DIR  = '/content/dataset'

if not os.path.exists(DATASET_DIR):
    print('Extracting dataset...')
    with zipfile.ZipFile(DATASET_ZIP, 'r') as z:
        z.extractall(DATASET_DIR)
    print(f'Dataset extracted to: {DATASET_DIR}')
else:
    print(f'Dataset already at: {DATASET_DIR}')

# ── Set up Python path ───────────────────────────────────────────────────────
os.chdir(PROJECT_DIR)
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

print(f'\nWorking dir: {os.getcwd()}')
print(f'Project files: {os.listdir(PROJECT_DIR)}')

import torch
print(f'\nPyTorch: {torch.__version__}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
else:
    print('WARNING: No GPU! Go to Runtime → Change runtime type → T4 GPU')

## Step 2 — Verify Dataset

In [ ]:
import os
from config import TRAIN_IMG_DIR, VALID_IMG_DIR, TEST_IMG_DIR, CLASS_NAMES, DATA_ROOT

print(f'DATA_ROOT: {DATA_ROOT}')
print(f'Classes: {CLASS_NAMES}')
print(f'Train images: {len(os.listdir(TRAIN_IMG_DIR))}')
print(f'Valid images: {len(os.listdir(VALID_IMG_DIR))}')
print(f'Test  images: {len(os.listdir(TEST_IMG_DIR))}')

## Step 3 — Verify Architecture

In [ ]:
import torch
from model import YOLOv1
from config import NUM_CLASSES, B, S

model = YOLOv1(num_classes=NUM_CLASSES, num_boxes=B, grid_size=S)
total_params = sum(p.numel() for p in model.parameters())
print(f'YOLOv1 — Parameters: {total_params:,}')

dummy = torch.randn(1, 3, 448, 448)
out = model(dummy)
print(f'Input:  {dummy.shape}')
print(f'Output: {out.shape}  (expected: [1, 7, 7, 14])')
assert out.shape == (1, 7, 7, 14), f'Shape mismatch: {out.shape}'
print('Architecture OK!')

## Step 4 — Visualize a Training Sample

In [ ]:
from dataset import YOLODataset
from config import TRAIN_IMG_DIR, TRAIN_LABEL_DIR
from visualize import visualize_sample
from IPython.display import display

ds = YOLODataset(TRAIN_IMG_DIR, TRAIN_LABEL_DIR, augment=False)
print(f'Dataset size: {len(ds)}')

img, target = ds[0]
print(f'Image shape: {img.shape}, Target shape: {target.shape}')
n_obj = (target[:, :, 4] > 0).sum().item()
print(f'Objects in first sample: {n_obj}')

vis_img = visualize_sample(ds, 0, '/content/sample_vis.jpg')
display(vis_img)

## Step 5 — Train YOLOv1

In [ ]:
import os, time, torch
from torch.utils.data import DataLoader
from config import (
    TRAIN_IMG_DIR, TRAIN_LABEL_DIR,
    VALID_IMG_DIR, VALID_LABEL_DIR,
    BATCH_SIZE, MOMENTUM, WEIGHT_DECAY, NUM_EPOCHS,
    LR_WARMUP_EPOCHS, LR_INIT, LR_MAIN, LR_SCHEDULE,
    CHECKPOINT_DIR, S, B, NUM_CLASSES,
)
from model import YOLOv1
from loss import YOLOv1Loss
from dataset import YOLODataset
from train import get_lr, train_one_epoch, validate

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Training on: {device}')

train_dataset = YOLODataset(TRAIN_IMG_DIR, TRAIN_LABEL_DIR, augment=True)
valid_dataset = YOLODataset(VALID_IMG_DIR, VALID_LABEL_DIR, augment=False)
print(f'Train: {len(train_dataset)}, Valid: {len(valid_dataset)}')

effective_batch = min(BATCH_SIZE, len(train_dataset))
train_loader = DataLoader(train_dataset, batch_size=effective_batch, shuffle=True,
                          num_workers=2, pin_memory=True, drop_last=True)
valid_loader = DataLoader(valid_dataset, batch_size=effective_batch, shuffle=False,
                          num_workers=2, pin_memory=True)

model = YOLOv1(num_classes=NUM_CLASSES, num_boxes=B, grid_size=S).to(device)
criterion = YOLOv1Loss()
optimizer = torch.optim.SGD(model.parameters(), lr=LR_INIT,
                             momentum=MOMENTUM, weight_decay=WEIGHT_DECAY)

os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print(f'Checkpoints: {CHECKPOINT_DIR}')
print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')

In [ ]:
best_mAP = 0.0
train_losses, val_losses, mAP_history = [], [], []

for epoch in range(NUM_EPOCHS):
    t_start = time.time()

    lr = get_lr(epoch)
    for pg in optimizer.param_groups:
        pg['lr'] = lr

    train_loss = train_one_epoch(model, train_loader, criterion, optimizer, device, epoch)
    train_losses.append(train_loss)

    val_loss, mAP, per_class_ap = 0.0, 0.0, {}
    if (epoch + 1) % 5 == 0 or epoch == NUM_EPOCHS - 1:
        val_loss, mAP, per_class_ap = validate(model, valid_loader, criterion, device)
        val_losses.append(val_loss)
        mAP_history.append((epoch + 1, mAP))

    elapsed = time.time() - t_start
    print(f'Epoch [{epoch+1}/{NUM_EPOCHS}]  LR: {lr:.6f}  '
          f'Train Loss: {train_loss:.4f}  Val Loss: {val_loss:.4f}  '
          f'mAP: {mAP:.4f}  Time: {elapsed:.1f}s')

    if per_class_ap:
        for cls_name, ap in per_class_ap.items():
            print(f'  {cls_name}: AP={ap:.4f}')

    if mAP > best_mAP:
        best_mAP = mAP
        torch.save({'epoch': epoch, 'model_state_dict': model.state_dict(),
                    'optimizer_state_dict': optimizer.state_dict(),
                    'mAP': mAP, 'train_loss': train_loss},
                   os.path.join(CHECKPOINT_DIR, 'yolov1_best.pth'))
        print(f'  >> Saved best model (mAP={mAP:.4f})')

    if (epoch + 1) % 25 == 0:
        torch.save({'epoch': epoch, 'model_state_dict': model.state_dict()},
                   os.path.join(CHECKPOINT_DIR, f'yolov1_epoch{epoch+1}.pth'))

torch.save({'epoch': NUM_EPOCHS - 1, 'model_state_dict': model.state_dict(), 'mAP': mAP},
           os.path.join(CHECKPOINT_DIR, 'yolov1_final.pth'))
print(f'\nTraining complete! Best mAP: {best_mAP:.4f}')

## Step 6 — Plot Training Curves

In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(range(1, len(train_losses) + 1), train_losses, label='Train Loss')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss')
ax1.set_title('Training Loss'); ax1.legend(); ax1.grid(True)

if mAP_history:
    epochs_eval, maps = zip(*mAP_history)
    ax2.plot(epochs_eval, maps, 'o-', color='green', label='mAP@0.5')
    ax2.set_xlabel('Epoch'); ax2.set_ylabel('mAP')
    ax2.set_title('Validation mAP@0.5'); ax2.legend(); ax2.grid(True)

plt.tight_layout()
plt.savefig('/content/training_curves.png', dpi=150)
plt.show()

## Step 7 — Evaluate on Validation Set

In [ ]:
from evaluate import evaluate as full_evaluate
from config import CONF_THRESHOLD, NMS_THRESHOLD

best_path = os.path.join(CHECKPOINT_DIR, 'yolov1_best.pth')
ckpt = torch.load(best_path, map_location=device, weights_only=False)
model.load_state_dict(ckpt['model_state_dict'])
print(f'Loaded best checkpoint from epoch {ckpt["epoch"] + 1}')

val_loss, mAP, per_class_ap = full_evaluate(
    model, valid_loader, criterion, device,
    conf_thresh=0.001, nms_thresh=NMS_THRESHOLD
)

print(f'\nValidation Results:')
print(f'  Loss:    {val_loss:.4f}')
print(f'  mAP@0.5: {mAP:.4f}')
print(f'\nPer-class AP:')
for cls_name, ap in per_class_ap.items():
    print(f'  {cls_name:12s}: {ap:.4f}')

## Step 8 — Detection on Test Images

In [ ]:
import glob
import matplotlib.pyplot as plt
from detect import detect_single
from config import TEST_IMG_DIR

test_images = sorted(glob.glob(os.path.join(TEST_IMG_DIR, '*.jpg')))[:6]
print(f'Showing detections on {len(test_images)} test images')

model.eval()
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.flatten()

for idx, img_path in enumerate(test_images):
    dets, result_img = detect_single(model, img_path, device)
    n_dets = dets.size(0) if dets.numel() > 0 else 0
    axes[idx].imshow(result_img)
    axes[idx].set_title(f'{os.path.basename(img_path)[:25]}... ({n_dets} dets)')
    axes[idx].axis('off')

plt.tight_layout()
plt.savefig('/content/test_detections.png', dpi=150)
plt.show()

## Step 9 — Save Results to Drive

In [ ]:
import shutil

# Save outputs to Google Drive
DRIVE_OUT = '/content/drive/MyDrive/YOLOv1_Results'
os.makedirs(DRIVE_OUT, exist_ok=True)

# Copy checkpoints
shutil.copytree(CHECKPOINT_DIR, f'{DRIVE_OUT}/checkpoints', dirs_exist_ok=True)

# Copy plots
for f in ['training_curves.png', 'test_detections.png']:
    if os.path.exists(f'/content/{f}'):
        shutil.copy(f'/content/{f}', f'{DRIVE_OUT}/{f}')

print(f'Results saved to Google Drive: {DRIVE_OUT}')
print(f'Best mAP: {best_mAP:.4f}')